SETUP

In [22]:
import pandas as pd

run_id_main = 48870
run_id_comp = 46654
file_main = f"run/{run_id_main}/results_{run_id_main}.tsv"
file_comp = f"run/{run_id_comp}/results_{run_id_comp}.tsv"

In [23]:
# main dataframe
df = pd.read_csv(file_main, sep='\t', header=0)
df.head()

,ID,max_iters,hash
0,00063d88244921d6ec46aeab6866a8e2,6,90add37784b6aad5
1,00063d88244921d6ec46aeab6866a8e2,20,2013f536f71dca4e
2,00072cf107ae1349c8c59a15c5ce4af1,6,bcb8e1607a5b7e9b
3,00072cf107ae1349c8c59a15c5ce4af1,20,bcb8e1607a5b7e9b
4,00076733bdbce94d7e44eca84f1425f0,6,50fb3c0e85ed461e


In [17]:
# dataframe for comparisons
df_comp = pd.read_csv(file_comp, sep='\t', header=0)
df_comp.head()

,ID,isohash
0,00063d88244921d6ec46aeab6866a8e2,fd5f9925c23132a6988d6b883cabf95d
1,00072cf107ae1349c8c59a15c5ce4af1,55060e6aa0f7a06dc70e163f2296eb95
2,00076733bdbce94d7e44eca84f1425f0,465f956423b4a18f2a53183fe4b5f682
3,000781b7a545fe723159e53127aff659,933f4c028a325d21969a109f37452e78
4,000a41cdca43be89ed62ea3abf2d0b64,fd393fdabc379273b867d42bc2be0ff4


HASH SUMMARY

In [24]:
hash_col = 'hash' if 'hash' in df.columns else 'isohash'

total = len(df)
na = df[hash_col].isna().sum()
unique = df[hash_col].nunique(dropna=True)
same = total - unique - na

print(f"TOTAL: {total}")
print(f"UNIQUE: {unique}")
print(f"NA: {na}")
print(f"SAME: {same}")

if 'max_iters' in df.columns:
    g = df.groupby('max_iters')[hash_col]
    s = pd.DataFrame({
        'TOTAL': g.size(),
        'NA': g.apply(lambda x: x.isna().sum()),
        'UNIQUE': g.nunique(dropna=True)
    })
    s['SAME'] = s['TOTAL'] - s['UNIQUE'] - s['NA']
    print("\nBY max_iters:")
    print(s.to_string())

TOTAL: 2000
UNIQUE: 1714
NA: 2
SAME: 284

BY max_iters:
           TOTAL  NA  UNIQUE  SAME
max_iters                         
6           1000   0     999     1
20          1000   2     997     1


COMPARE MAX_ITERS WITH EACH OTHER

GET ALL DUPLICATE HASHES AND THEIR INSTANCES

In [9]:
# adding isohash to duplicated hashes of main to compare
dupe_rows = df[df['hash'].duplicated(keep=False)].sort_values('hash')
merged = dupe_rows.merge(
    df_comp[['ID','isohash']],
    on='ID',
    how='left'
)
print(merged.head(10))

                                 ID              hash  \
0  5c257009a97cdac455df115b916abf45  024ba17ab580643e   
1  eb34dcb040e9956397c306400da096f1  024ba17ab580643e   
2  bb564b1160d0bc9032538da7304a7a7b  0511ff26702ea55c   
3  c1b30b4e03c3024ad9d084e29e79aa46  0511ff26702ea55c   
4  63bd4dfe32c730303164e65d49783a77  08a6650f25c656dc   
5  4ca24eacb14a3f2b898e900f3b1b6862  08a6650f25c656dc   
6  37382b66a104f6387b5750bf51942996  0a8740a4f410272b   
7  83628f8fe950fde6e67a24f9eb18c884  0a8740a4f410272b   
8  75e0c8b167b220c2aecb64c9e265c949  0a961b4d037cae84   
9  f3b8c2cb6c35259cbaf7416d6e9b02c7  0a961b4d037cae84   

                            isohash  
0  2807e48f13872c6f440e6c22d8ea63d7  
1  2807e48f13872c6f440e6c22d8ea63d7  
2  d30963ec40ca6987639834c86206dbdf  
3  d30963ec40ca6987639834c86206dbdf  
4  3fa1086a7d653996887e11a6ce900315  
5  3fa1086a7d653996887e11a6ce900315  
6  459776914cf49261bfe3dba1a8f902ee  
7  459776914cf49261bfe3dba1a8f902ee  
8  5e0067ce01aa525ec5e47dc6502

In [21]:
# adding hash of main to duplicated hashes of isohash
dupe_rows_comp = df_comp[df_comp['isohash'].duplicated(keep=False)].sort_values('isohash')
merged = dupe_rows_comp.merge(
    df[['ID','hash']],
    on='ID',
    how='left'
)
print(merged.head(10))

                                 ID                           isohash hash
0  4666906deb05406c0ecc18de81673d76  00a1a353e84e203de503f206c02b7a2c  NaN
1  ae07d3502702cd744ad48e254abb192f  00a1a353e84e203de503f206c02b7a2c  NaN
2  74acaa627ddc5fdab40d62172f56a827  00c5b384d9311d83da0be53e990b4904  NaN
3  66ab346df72c5effa292342037bc3909  00c5b384d9311d83da0be53e990b4904  NaN
4  df3da7a967605e804571a36c099416b7  00efc6678bd4710b52c3448cc6e9c5da  NaN
5  847df6bb7bd0a8fcedb57e012cfe0014  00efc6678bd4710b52c3448cc6e9c5da  NaN
6  56270bc03e3792c1c24d0deb72f75bbb  00efc6678bd4710b52c3448cc6e9c5da  NaN
7  d93819b0aaf7569fe0b28e86db542330  00efc6678bd4710b52c3448cc6e9c5da  NaN
8  ac0d8f93857783ae914af63f97bc33b3  00efc6678bd4710b52c3448cc6e9c5da  NaN
9  f1c7fb2d19b357ee0ed4e9df28a1b533  012ae103f8674cab1b6a5721318dad86  NaN
